[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/14_indexacion_ann/14_indexacion_ann.ipynb)

# Módulo 14 · Indexación y búsqueda aproximada de vecinos (ANN)

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) · **GPU:** no necesaria (CPU; todo en `FAST_DEV_RUN` en minutos) ·
**Cómputo estimado:** 0 unidades de Colab (CPU); con `FAST_DEV_RUN = False` conviene un runtime con más RAM, pero no GPU.

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 3 h): secciones 1–5 (fuerza bruta, métricas, IVF, PQ, HNSW) y 10 (FAISS en uso real). Ampliación (≈ 1–2 h): Vamana/DiskANN (6), ScaNN (7), RaBitQ (8) y búsqueda filtrada (9).

**Prerrequisitos:** módulo 12 (dimensión, Matryoshka, cuantización int8/binaria) y 13 (qué vectores indexas: densos, multi-vector).
**Puente con recsys:** el módulo [`recsys-course/08_deep_retrieval_ann`](../../recsys-course/08_deep_retrieval_ann/README.md) ya enseña FAISS (Flat/IVF/IVF-PQ/HNSW) con curvas recall-latencia sobre vectores de *two-tower*. Aquí **no lo repetimos**: construimos los algoritmos desde cero (mini-HNSW, PQ/OPQ, Vamana, RaBitQ), tratamos el recall como métrica de ingeniería y añadimos lo que allí no está: **grafos en disco, cuantización con garantías y búsqueda filtrada**.

**Hilo conductor — Políglota.** Con 10 M de fragmentos, la búsqueda exacta cuesta ~40 GB de lecturas por consulta. El CTO pide una *ficha técnica* del índice: **recall@10 ≥ 0.95, p99 ≤ 20 ms, memoria ≤ X GB**. Hoy aprendes a construirla con datos.

> **Estado del arte a octubre de 2026.** Los algoritmos clásicos son 🟢; cualquier cifra de adopción por empresas o de rendimiento relativo está fechada y con fuente. La operación de bases vectoriales (sharding, réplicas, actualizaciones) es el módulo 18.

## 🎯 Objetivos de aprendizaje
1. **Medir** un índice ANN con el protocolo correcto: neighbor recall@k contra fuerza bruta, latencia p50/p99 con batch=1 y memoria real.
2. **Implementar desde cero** IVF (k-means + listas invertidas), PQ/OPQ (cuantización de producto con ADC) y comparar con FAISS.
3. **Implementar un mini-HNSW** (capas, `efConstruction`, heurística de selección de vecinos, búsqueda voraz por capas) y razonar sobre sus parámetros `M`, `ef`.
4. **Implementar Vamana** (la base de DiskANN) con `RobustPrune` y explicar por qué el grafo se diseña para leer pocas páginas de SSD.
5. **Derivar la idea de ScaNN** (cuantización anisótropa) y la de **RaBitQ** (estimador insesgado con cota de error) e implementar una versión mini de RaBitQ.
6. **Resolver búsqueda filtrada**: pre-filtro, post-filtro y filtro dentro del grafo, y medir cómo se rompe el recall.
7. **Construir y comparar índices FAISS reales** y elegir la configuración que cumple un SLA (recall, latencia, memoria).

## 🔁 Conexión con módulos anteriores
<details><summary><b>1 · (M12)</b> ¿Qué memoria ocupan 10 M de vectores de 1024 dims en float32, int8 y binario?</summary>

40.96 GB, 10.24 GB y 1.28 GB. La cuantización de *vectores* (módulo 12) es un caso particular de lo que hoy hacemos *dentro del índice*: PQ y RaBitQ son cuantizaciones **diseñadas para que la distancia aproximada se calcule sin descomprimir**.
</details>
<details><summary><b>2 · (M12)</b> ¿Por qué se renormaliza tras truncar y qué métrica de similitud usan casi todos los embedders?</summary>

Para volver a la esfera unidad; así producto interno = coseno = (monótona de) distancia L2. Usaremos vectores normalizados, de modo que MIPS, coseno y L2 dan el **mismo ranking**.
</details>
<details><summary><b>3 · (M13)</b> ¿Qué es lo más caro de almacenar en late interaction y cómo lo mitiga MUVERA?</summary>

Un vector por token. MUVERA construye un único vector de dimensión fija por documento (FDE) para poder usar **un índice ANN estándar** (este módulo) como generador de candidatos y re-puntuar con MaxSim exacto.
</details>
<details><summary><b>4 · (recsys 08)</b> ¿Qué métrica compara un índice aproximado con el exacto?</summary>

El *recall* del top-K aproximado respecto al top-K exacto (neighbor recall). Es distinto del recall de la **tarea** (¿está el documento relevante?): el primero mide solo el error del índice.
</details>

## 🗺️ Mapa de madurez (a oct-2026)
🟢 sólido · 🟡 emergente · 🔴 experimental.

| Técnica | Madurez | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| Fuerza bruta (exacta, SIMD/GPU) | 🟢 | línea base obligatoria; óptima hasta cientos de miles de vectores | — |
| IVF (listas invertidas por k-means) | 🟢 | FAISS, Milvus, muchos sistemas | Jégou et al. 2011; Douze et al. 2024 |
| PQ / OPQ | 🟢 | FAISS (IVF-PQ), ScaNN, DiskANN (PQ en RAM) | Jégou et al. 2011; Ge et al. 2013 |
| HNSW | 🟢 | Lucene/Elasticsearch/OpenSearch, pgvector, Qdrant, Weaviate, FAISS… | Malkov & Yashunin 2016 (1603.09320) |
| ScaNN (cuantización anisótropa) | 🟢 | Google (`scann`); base de la búsqueda vectorial de Google | Guo et al. 2019 (1908.10396) |
| DiskANN / Vamana (grafo + SSD) | 🟢/🟡 | Microsoft (código abierto); sistemas con índices en disco | Subramanya et al. 2019; FreshDiskANN 2105.09613 |
| RaBitQ / Extended-RaBitQ | 🟡 | Elastic (BBQ, inspirado en RaBitQ), Milvus ≥ 2.6, contribuciones a FAISS | Gao & Long 2024 (2405.12497); 2409.09913 |
| Cuantización binaria + rescoring | 🟡 | Elastic BBQ, proveedores de embeddings (módulo 12) | blog HF 2024; Elastic |
| Búsqueda filtrada dentro del grafo (ACORN y similares) | 🟡 | investigación + algunos motores | Patel et al. 2024 (2403.04871) |
| GPU para ANN (FAISS-GPU/cuVS, CAGRA) | 🟡 | NVIDIA/Meta; depende de versiones | Douze et al. 2024; docs |

## 0 · Preparación
Esta lección usa **vectores sintéticos "tipo embedding"** (mezcla de clusters con espectro decreciente, normalizados) para poder generar cientos de miles de vectores sin descargar nada, y al final repite una medición con **vectores reales de Políglota** (módulo 12) si hay un embedder disponible. Todos los experimentos usan **un hilo**: la latencia de una query es lo que importa.

In [ ]:
!pip install -q -U faiss-cpu numpy scipy scikit-learn matplotlib pandas sentence-transformers

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
import faiss
faiss.omp_set_num_threads(N_THREADS)         # medición justa: un hilo (la latencia de UNA query es lo que importa)
import pandas as pd
import matplotlib.pyplot as plt

def exacto_topk(Q, X, k=10):
    """Fuerza bruta: el 'oráculo' para medir recall. Q (nq,d), X (n,d) normalizados -> índices (nq,k)."""
    out = np.empty((len(Q), k), dtype=np.int64)
    for s in range(0, len(Q), 64):
        S = Q[s:s + 64] @ X.T
        idx = np.argpartition(-S, k, axis=1)[:, :k]
        out[s:s + 64] = np.take_along_axis(idx, np.take_along_axis(S, idx, 1).argsort(1)[:, ::-1], 1)
    return out

def recall_vs_gt(idx, gt):
    """Neighbor recall@k: fracción del top-k exacto que devuelve el índice aproximado (ignora los -1 de FAISS)."""
    return float(np.mean([len(set(a[a >= 0].tolist()) & set(b.tolist())) / len(b) for a, b in zip(idx, gt)]))

def mide_latencia(fn, Q, calentamiento=10, repeticiones=1):
    """Latencia por query con batch=1 (así se sirve en producción). fn(q: (d,)) -> cualquier cosa. Devuelve p50, p99 (ms) y QPS."""
    for q in Q[:calentamiento]:
        fn(q)
    t = []
    for _ in range(repeticiones):
        for q in Q:
            t0 = time.perf_counter(); fn(q); t.append(time.perf_counter() - t0)
    t = np.array(t) * 1e3
    return float(np.percentile(t, 50)), float(np.percentile(t, 99)), float(1e3 / t.mean())

def tam_indice_MB(index):
    """Tamaño real serializado del índice FAISS en MB (incluye vectores, listas y grafo)."""
    return faiss.serialize_index(index).nbytes / 1e6

In [ ]:
def genera_vectores(n, d, nc=30, nq=200, decay=0.2, sep=0.5, seed=0):
    """Vectores sintéticos 'tipo embedding': mezcla de nc clusters con espectro decreciente (unas direcciones dominan),
    normalizados a norma 1. decay=0 -> casi isótropo (difícil para ANN); decay alto -> muy anisótropo (IVF/HNSW lo separan mejor, pero la PQ pierde recall porque las diferencias entre vecinos son pequeñas frente a su error de cuantización)."""
    rng = np.random.default_rng(seed)
    spec = 1.0 / (1 + np.arange(d)) ** decay
    spec /= spec.max()
    centros = rng.normal(size=(nc, d)) * spec * sep
    a = rng.integers(0, nc, size=n + nq)
    V = centros[a] + rng.normal(size=(n + nq, d)) * spec
    V /= np.linalg.norm(V, axis=1, keepdims=True)
    return V[:n].astype(np.float32), V[n:].astype(np.float32)

N = 20000 if FAST_DEV_RUN else 200000
D = 64 if FAST_DEV_RUN else 128
X, Q = genera_vectores(N, D)
t0 = time.perf_counter(); GT = exacto_topk(Q, X, 10); t_gt = time.perf_counter() - t0
print(f"N={N:,} · d={D} · {len(Q)} queries · fuerza bruta de todas: {t_gt:.2f}s · {t_gt/len(Q)*1e3:.2f} ms/query (batch)")

## 1 · Intuición: por qué existe el ANN y cuándo NO usarlo

Buscar los $k$ vectores más parecidos a una query entre $N$ vectores de dimensión $d$ cuesta $O(N d)$ operaciones y, sobre todo, **leer $4Nd$ bytes de memoria** por consulta: es un problema limitado por el ancho de banda. Con $N=10^7$ y $d=1024$ son **41 GB por consulta**. Los índices ANN cambian exactitud por velocidad con tres ideas ortogonales:

| Idea | Familia | Qué evita |
|---|---|---|
| **Particionar** el espacio y mirar pocas particiones | IVF | leer todos los vectores |
| **Navegar un grafo** de proximidad | HNSW, Vamana/DiskANN | comparar con todos los vectores |
| **Comprimir** los vectores y calcular distancias aproximadas sobre los códigos | PQ, ScaNN, RaBitQ, binaria | leer y multiplicar vectores completos |

Se combinan (IVF-PQ, HNSW+PQ, DiskANN con PQ en RAM). Y el triángulo de siempre: **recall – latencia – memoria**; mejorar dos suele empeorar el tercero.

> 💡 **Analogía con IR y recsys.** IVF ≈ un índice invertido sobre "celdas" en vez de términos. HNSW ≈ "seis grados de separación" con autopistas (capas altas) y calles locales (capa 0). PQ ≈ compresión con pérdida tipo JPEG: guarda códigos y reconstruye aproximadamente. En recsys (módulo 08) el índice de ítems es la misma pieza.

> ⚠️ **Cuándo NO usar ANN.** Hasta ~$10^5$ vectores, la **fuerza bruta** exacta (BLAS/SIMD) suele ser igual de rápida y sin pérdida de recall. Empieza siempre por medirla.

> ### 🎨 GEMINI-IMAGEN · ID: `M14-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Mapa de familias de índices ANN**
>
> **Qué muestra:** Las tres ideas (particionar, navegar un grafo, comprimir) y cómo se combinan en los índices que se usan en producción.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres columnas: particionar con celdas, navegar un grafo por capas y comprimir vectores con códigos
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible. Sin ilustraciones decorativas ni texto inventado.
> 
> Título arriba: "Tres ideas para buscar vecinos sin mirar todo".
> Tres columnas con encabezado coloreado:
> 1) Azul "Particionar": dibujo 2D de un plano dividido en celdas de Voronoi con puntos; una consulta marcada con una estrella cae en una celda, y se resaltan 2 celdas vecinas (nprobe = 2). Etiqueta: "IVF".
> 2) Naranja "Navegar un grafo": tres capas horizontales apiladas (capa 2 arriba con pocos nodos, capa 1, capa 0 abajo con muchos nodos), una flecha que baja saltando entre nodos hacia la estrella. Etiquetas: "HNSW" y "Vamana / DiskANN".
> 3) Verde "Comprimir": un vector de 8 cuadros de colores que se parte en 4 pares y cada par se sustituye por un número de código (por ejemplo 17, 203, 5, 88). Etiquetas: "PQ", "ScaNN", "RaBitQ (1 bit por dimensión)".
> Debajo de las columnas, una barra horizontal gris con flechas que unen las columnas, y el texto: "Combinaciones habituales: IVF-PQ · HNSW + PQ · DiskANN (grafo en SSD + PQ en RAM)".
> Pie de figura pequeño: "Triángulo de diseño: recall · latencia · memoria".
> 
> Evita: texto ilegible, más de tres columnas, flechas cruzadas confusas, logos de empresas, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M14-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres columnas: particionar con celdas, navegar un grafo por capas y comprimir vectores con códigos](assets/M14-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> PARTICIONAR (IVF)      NAVEGAR (HNSW / Vamana)       COMPRIMIR (PQ / ScaNN / RaBitQ)
> ┌──┬──┬──┐ ★ nprobe=2  capa2:  o       o              [■■□□▪▪▫▫] -> [17|203|5|88]
> ├──┼──┼──┤             capa1:  o  o  o  o                    códigos de 1 byte
> └──┴──┴──┘             capa0:  o o o o o o o ★
> Combinaciones: IVF-PQ · HNSW+PQ · DiskANN (grafo en SSD + PQ en RAM)
> ```

> 👀 **Qué debes observar.** Las tres columnas atacan costes distintos: **cuántos vectores se tocan** (IVF), **cuántas comparaciones se hacen** (grafo) y **cuántos bytes ocupa y se lee cada vector** (compresión). Los sistemas reales combinan una de cada tipo.

## 2 · Fuerza bruta y el protocolo de medición

Antes de cualquier índice hace falta (a) el **oráculo** (top-$k$ exacto), (b) la métrica y (c) un protocolo de latencia honesto:

- **Neighbor recall@$k$** $=\frac{|\text{ANN}_k\cap\text{Exacto}_k|}{k}$, promediado sobre queries. Mide el error del índice, no la calidad de la tarea (para eso, nDCG, módulos 12–13 y 15).
- **Latencia con batch=1** y percentiles **p50/p99** (la media esconde las colas). Calienta antes; un hilo; mismo hardware para todas las configuraciones.
- **Memoria real** del índice (bytes serializados), no la estimada a ojo.

In [ ]:
rng = np.random.default_rng(SEED)
tamanos = [int(x) for x in np.geomspace(1000, N, 6)]
filas = []
for n_ in tamanos:
    Xn = X[:n_]
    flat = faiss.IndexFlatIP(D); flat.add(Xn)
    p50_np, _, _ = mide_latencia(lambda q: Xn @ q, Q[:100])                  # NumPy: producto matriz-vector (BLAS)
    p50_fa, p99_fa, _ = mide_latencia(lambda q: flat.search(q[None], 10), Q[:100])
    filas.append((n_, p50_np, p50_fa, p99_fa))
bf = pd.DataFrame(filas, columns=["N", "numpy p50 (ms)", "faiss Flat p50 (ms)", "faiss Flat p99 (ms)"])
display(bf.round(3))
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].loglog(bf["N"], bf["numpy p50 (ms)"], "o-", label="NumPy X @ q", color="#F28E2B")
axs[0].loglog(bf["N"], bf["faiss Flat p50 (ms)"], "s-", label="FAISS IndexFlatIP", color="#1F6FEB")
axs[0].loglog(bf["N"], bf["faiss Flat p99 (ms)"], "s:", label="FAISS Flat (p99)", color="#1F6FEB", alpha=.6)
axs[0].set_xlabel("N (vectores)"); axs[0].set_ylabel("ms por query (batch=1)"); axs[0].set_title("La fuerza bruta escala linealmente en N"); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3, which="both")
gb_s = 4 * D * bf["N"] / (bf["faiss Flat p50 (ms)"] * 1e-3) / 1e9
axs[1].semilogx(bf["N"], gb_s, "o-", color="#2CA02C")
axs[1].set_xlabel("N (vectores)"); axs[1].set_ylabel("GB/s efectivos leídos"); axs[1].set_title("Ancho de banda efectivo de la fuerza bruta"); axs[1].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()
print(f"A N={N:,} y d={D}: {4*D*N/1e6:.0f} MB que leer por query. Para 10 M x 1024 d serían {4*1024*10_000_000/1e9:.0f} GB.")

> 👀 **Qué debes observar.** (1) En escala log-log la latencia es una **recta de pendiente 1**: duplicar $N$ duplica el tiempo. (2) A $N$ pequeño domina el *overhead* de llamada y el ancho de banda efectivo es bajo; al crecer $N$ se acerca al límite de memoria de tu CPU. (3) Con tu $N$ actual la fuerza bruta puede ser *rápida*: **el ANN solo gana cuando $N$ es grande o el presupuesto de latencia es estricto**. Guarda este gráfico como línea base.

## 3 · IVF desde cero: particionar con k-means

**Entrenamiento:** k-means con $n_{list}$ centroides (regla práctica: $n_{list}\approx\sqrt N$ a $4\sqrt N$). Cada vector se asigna a su centroide más cercano y va a su **lista invertida**. **Búsqueda:** calcula la similitud de la query con los $n_{list}$ centroides, visita las **$n_{probe}$ listas** más cercanas y busca exacto solo en ellas:

$$\text{coste} \approx n_{list}\,d + \frac{n_{probe}}{n_{list}}\,N\,d$$

Dos términos: el escaneo de centroides (crece con $n_{list}$) y el de las listas (crece con $n_{probe}$). El recall sube con $n_{probe}$ a costa de latencia.

In [ ]:
def kmeans(V, k, iters=12, seed=0):
    """k-means esférico (vectores normalizados: asignar por producto interno máximo, recentrar y normalizar)."""
    rng = np.random.default_rng(seed)
    C = V[rng.choice(len(V), k, replace=False)].copy()
    for _ in range(iters):
        a = (V @ C.T).argmax(1)
        for j in range(k):
            m = a == j
            if m.any():
                c = V[m].mean(0); C[j] = c / np.linalg.norm(c)
    return C

class MiniIVF:
    def __init__(self, nlist, seed=0):
        self.nlist, self.seed = nlist, seed
    def fit(self, V):
        self.C = kmeans(V, self.nlist, seed=self.seed)
        asign = (V @ self.C.T).argmax(1)
        self.V = V
        self.listas = [np.where(asign == j)[0] for j in range(self.nlist)]      # listas invertidas: celda -> ids
        return self
    def search(self, q, k=10, nprobe=4):
        celdas = np.argpartition(-(self.C @ q), nprobe)[:nprobe]
        cand = np.concatenate([self.listas[j] for j in celdas])
        s = self.V[cand] @ q
        top = cand[np.argsort(-s)[:k]]
        return top, len(cand) + self.nlist                                      # (ids, distancias calculadas)

NLIST = int(np.sqrt(N))
t0 = time.perf_counter(); ivf = MiniIVF(NLIST).fit(X); t_build = time.perf_counter() - t0
tam_listas = np.array([len(l) for l in ivf.listas])
print(f"nlist={NLIST} · entrenamiento+asignación {t_build:.1f}s · tamaño de lista: media {tam_listas.mean():.0f}, máx {tam_listas.max()}")

# FAISS equivalente
cuant = faiss.IndexFlatIP(D)
ivf_f = faiss.IndexIVFFlat(cuant, D, NLIST, faiss.METRIC_INNER_PRODUCT); ivf_f.train(X); ivf_f.add(X)
filas = []
for nprobe in (1, 2, 4, 8, 16, 32, 64):
    res = [ivf.search(q, 10, nprobe) for q in Q]
    ivf_f.nprobe = nprobe
    _, I = ivf_f.search(Q, 10)
    p50, p99, _ = mide_latencia(lambda q: ivf_f.search(q[None], 10), Q[:100])
    filas.append(dict(nprobe=nprobe, recall_propio=recall_vs_gt(np.stack([r[0] for r in res]), GT), recall_faiss=recall_vs_gt(I, GT),
                      pct_escaneado=100 * np.mean([r[1] for r in res]) / N, faiss_p50_ms=p50))
tab_ivf = pd.DataFrame(filas)
display(tab_ivf.round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].plot(tab_ivf["nprobe"], tab_ivf["recall_propio"], "o-", label="IVF desde cero", color="#F28E2B")
axs[0].plot(tab_ivf["nprobe"], tab_ivf["recall_faiss"], "s--", label="FAISS IndexIVFFlat", color="#1F6FEB")
axs[0].set_xscale("log", base=2); axs[0].set_xlabel("nprobe"); axs[0].set_ylabel("recall@10"); axs[0].set_title("Recall vs nprobe"); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(tab_ivf["pct_escaneado"], tab_ivf["recall_propio"], "o-", color="#F28E2B")
for r in tab_ivf.itertuples():
    axs[1].annotate(f"{r.nprobe}", (r.pct_escaneado, r.recall_propio), textcoords="offset points", xytext=(4, -10), fontsize=7)
axs[1].set_xlabel("% de vectores escaneados"); axs[1].set_ylabel("recall@10"); axs[1].set_title("Cuánto hay que mirar para un recall dado"); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Tu IVF y el de FAISS dan casi el mismo recall (pequeñas diferencias por la inicialización de k-means). (2) El recall **sube rápido y luego se aplana**: pasar de `nprobe` 16 a 64 cuesta 4× más escaneo para ganar pocos puntos. (3) A la derecha, el recall se pierde sobre todo por vecinos que caen **en una celda vecina**: cuanto menos separados los clusters (datos más "difíciles"), más celdas hay que mirar. Lo habitual en producción: elegir `nprobe` como el menor valor que cumple el recall objetivo en un conjunto de validación.

## 4 · Product Quantization (PQ) y OPQ desde cero

**PQ** (Jégou et al., 2011) divide cada vector en $m$ subvectores de dimensión $d/m$ y aprende **un diccionario de $K=256$ centroides por subespacio** (k-means). Cada vector se sustituye por $m$ índices de **1 byte** (compresión $4d/m$). La distancia entre una query $q$ y un vector codificado se calcula con **ADC** (*asymmetric distance computation*): se precomputa una tabla $T_j[c]=\langle q_j,\mu_{j,c}\rangle$ (para cada subespacio $j$ y centroide $c$) y el score es
$$\langle q,\hat x\rangle=\sum_{j=1}^{m}T_j\big[\text{code}_j(x)\big],$$
solo $m$ búsquedas en tabla y sumas por vector (la query **no** se cuantiza: asimétrica). **OPQ** (Ge et al., 2013) aprende además una **rotación ortogonal** $R$ que equilibra la varianza entre subespacios antes de cuantizar; se resuelve alternando "entrenar PQ" y "Procrustes" ($R=UV^\top$ con $U\Sigma V^\top=\text{SVD}(X^\top\hat X)$).

In [ ]:
def entrena_pq(V, m, K=256, iters=8, n_train=8000):
    d = V.shape[1] // m
    muestra = V[np.random.default_rng(SEED).choice(len(V), min(n_train, len(V)), replace=False)]
    return [kmeans_l2(muestra[:, j * d:(j + 1) * d], min(K, len(muestra)), iters, seed=j) for j in range(m)]

def kmeans_l2(V, k, iters, seed=0):
    rng = np.random.default_rng(seed)
    C = V[rng.choice(len(V), k, replace=False)].copy()
    for _ in range(iters):
        a = (-2 * V @ C.T + (C ** 2).sum(1)).argmin(1)
        for j in range(k):
            m_ = a == j
            if m_.any():
                C[j] = V[m_].mean(0)
    return C

def codifica_pq(V, libros):
    d = V.shape[1] // len(libros)
    return np.stack([(-2 * V[:, j * d:(j + 1) * d] @ c.T + (c ** 2).sum(1)).argmin(1) for j, c in enumerate(libros)], 1).astype(np.uint8)

def decodifica_pq(codigos, libros):
    return np.concatenate([c[codigos[:, j]] for j, c in enumerate(libros)], 1)

def adc(q, libros, codigos):
    """Tablas T_j[c] = <q_j, mu_jc> y score = suma_j T_j[code_j]: solo m accesos a tabla por vector."""
    d = len(q) // len(libros)
    T = [c @ q[j * d:(j + 1) * d] for j, c in enumerate(libros)]
    return sum(T[j][codigos[:, j]] for j in range(len(libros)))

filas, PQ = [], {}
for m in (4, 8, 16, 32):
    if D % m:
        continue
    t0 = time.perf_counter(); libros = entrena_pq(X, m); codigos = codifica_pq(X, libros); t_pq = time.perf_counter() - t0
    mse = float(((X - decodifica_pq(codigos, libros)) ** 2).sum(1).mean())
    r_pq = np.stack([np.argsort(-adc(q, libros, codigos))[:10] for q in Q])
    r_refine = []                                                   # PQ para candidatos + re-ranking exacto de los 100 mejores
    for q in Q:
        cand = np.argsort(-adc(q, libros, codigos))[:100]
        r_refine.append(cand[np.argsort(-(X[cand] @ q))[:10]])
    PQ[m] = (libros, codigos)
    filas.append(dict(m=m, bytes_por_vector=m, compresion=f"{4*D/m:.0f}x", mse=mse, recall_ADC=recall_vs_gt(r_pq, GT),
                      recall_ADC_mas_refine100=recall_vs_gt(np.stack(r_refine), GT), seg_entrenar=t_pq))
tab_pq = pd.DataFrame(filas)
display(tab_pq.round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].plot(tab_pq["bytes_por_vector"], tab_pq["recall_ADC"], "o-", label="solo ADC", color="#F28E2B")
axs[0].plot(tab_pq["bytes_por_vector"], tab_pq["recall_ADC_mas_refine100"], "s-", label="ADC + re-ranking exacto de 100", color="#2CA02C")
axs[0].axvline(4 * D, color="#6B7280", ls=":"); axs[0].text(4 * D, 0.05, " float32", color="#6B7280", fontsize=8)
axs[0].set_xscale("log", base=2); axs[0].set_xlabel("bytes por vector (m)"); axs[0].set_ylabel("recall@10"); axs[0].set_title("PQ: memoria vs recall"); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3)

# OPQ: dónde ayuda de verdad (varianza concentrada en pocas dimensiones)
Xs, _ = genera_vectores(6000, 64, nc=30, nq=10, decay=1.0, sep=1.0)
def mse_pq(V, m=8, K=64):
    lib = entrena_pq(V, m, K, iters=6, n_train=6000); return float(((V - decodifica_pq(codifica_pq(V, lib), lib)) ** 2).sum(1).mean())
Rm, mses = np.eye(64), []
mses.append(mse_pq(Xs))
for it in range(3):
    lib = entrena_pq(Xs @ Rm, 8, 64, iters=6, n_train=6000); Y = decodifica_pq(codifica_pq(Xs @ Rm, lib), lib)
    U, _, Vt = np.linalg.svd(Xs.T @ Y); Rm = U @ Vt
    mses.append(mse_pq(Xs @ Rm))
axs[1].plot(range(len(mses)), mses, "o-", color="#1F6FEB")
axs[1].set_xlabel("iteración de OPQ (0 = PQ sin rotación)"); axs[1].set_ylabel("error de cuantización (MSE)"); axs[1].set_title("OPQ reduce la distorsión"); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"MSE PQ = {mses[0]:.3f} → OPQ = {mses[-1]:.3f} ({100*(1-mses[-1]/mses[0]):.0f}% menos)")

> 👀 **Qué debes observar.** (1) Con pocos bytes por vector el ADC **solo** pierde mucho recall (los datos de esta lección son poco anisótropos, un caso difícil para PQ); con más bytes sube. (2) **PQ + re-ranking exacto** de unos pocos cientos de candidatos recupera casi todo el recall con el mismo índice comprimido: es el patrón estándar (IVF-PQ + *refine*). Por eso en producción los códigos PQ viven en RAM y los vectores completos en disco/SSD. (3) A la derecha, **OPQ** reduce el error de cuantización cuando la varianza está desigualmente repartida entre subespacios (aquí `decay=1.0`); en datos casi isótropos la ganancia es mínima. Mide antes de pagar su coste de entrenamiento.

> ### 🎨 GEMINI-IMAGEN · ID: `M14-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Product Quantization: del vector al código de m bytes**
>
> **Qué muestra:** El proceso de PQ: partir en subvectores, buscar el centroide más cercano en cada diccionario y guardar solo los índices; y cómo se calcula la distancia con tablas (ADC).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Un vector dividido en cuatro subvectores que se reemplazan por cuatro índices de diccionario y se puntúan con tablas
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif. Sin decoración.
> 
> PARTE SUPERIOR (codificar), de izquierda a derecha:
> 1) Una barra gris con 8 casillas etiquetada "vector x (d = 8, ejemplo)". Se divide con líneas verticales en 4 trozos de 2 casillas, coloreados azul, naranja, verde, gris, con la etiqueta "m = 4 subvectores".
> 2) Cuatro cajas pequeñas (una por subvector), cada una con el texto "diccionario de 256 centroides" y una flecha desde el subvector hasta un centroide marcado.
> 3) A la derecha, cuatro casillas con números: "17 | 203 | 5 | 88" bajo el título "código PQ = 4 bytes".
> PARTE INFERIOR (buscar con ADC): una query q dividida en los mismos 4 trozos; debajo, 4 tablas pequeñas "T1 … T4" (cada una con 256 valores, mostrar solo tres valores y puntos suspensivos); un recuadro final con la fórmula "score = T1[17] + T2[203] + T3[5] + T4[88]" y la etiqueta "4 accesos a tabla por vector".
> Pie de figura: "d = 1024 con m = 64 → 64 bytes por vector (64× menos que float32)".
> 
> Evita: texto ilegible, números distintos de los indicados, flechas cruzadas, logos, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M14-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Un vector dividido en cuatro subvectores que se reemplazan por cuatro índices de diccionario y se puntúan con tablas](assets/M14-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> x = [x1 x2 | x3 x4 | x5 x6 | x7 x8]      m = 4 subvectores
>       ↓        ↓        ↓        ↓       (centroide más cercano en cada diccionario de 256)
> código = [ 17 | 203 | 5 | 88 ]           (4 bytes)
> score(q, x) ≈ T1[17] + T2[203] + T3[5] + T4[88]       Tj[c] = <q_j, centroide_jc>
> ```

> 👀 **Qué debes observar.** La query nunca se comprime: se construyen $m$ tablas de $K$ valores **una vez por query** y cada documento cuesta $m$ lecturas de tabla. La compresión es del lado del índice; la precisión del lado de la query (por eso se llama *asimétrico*).

### Combinar: IVF-PQ y su memoria con FAISS
Las dos ideas se combinan: IVF reduce **cuántos** vectores miras y PQ **cuánto cuesta** mirar cada uno. En FAISS, `IndexIVFPQ` codifica el **residuo** (vector − centroide de su celda), que tiene menos energía y se cuantiza mejor. Veamos el tamaño real serializado de cada familia con el mismo $N$.

In [ ]:
def construye_faiss(clave):
    if clave == "Flat":
        ix = faiss.IndexFlatIP(D)
    elif clave == "IVFFlat":
        ix = faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, NLIST, faiss.METRIC_INNER_PRODUCT)
    elif clave == "IVFPQ":
        ix = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, NLIST, 16, 8, faiss.METRIC_INNER_PRODUCT)
    elif clave == "HNSW":
        ix = faiss.IndexHNSWFlat(D, 16, faiss.METRIC_INNER_PRODUCT); ix.hnsw.efConstruction = 100
    if not ix.is_trained:
        ix.train(X)
    t0 = time.perf_counter(); ix.add(X); return ix, time.perf_counter() - t0

filas, IDX = [], {}
for clave in ("Flat", "IVFFlat", "IVFPQ", "HNSW"):
    ix, t_add = construye_faiss(clave)
    IDX[clave] = ix
    filas.append(dict(indice=clave, MB=tam_indice_MB(ix), bytes_por_vector=tam_indice_MB(ix) * 1e6 / N, seg_construir=t_add))
mem = pd.DataFrame(filas)
display(mem.round(2))
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.bar(mem["indice"], mem["bytes_por_vector"], color=["#6B7280", "#1F6FEB", "#2CA02C", "#F28E2B"])
ax.axhline(4 * D, color="k", ls=":"); ax.text(0.02, 4 * D + 4, f"float32 = {4*D} B/vector", fontsize=8)
ax.set_ylabel("bytes por vector (serializado)"); ax.set_title(f"Memoria real por familia · N={N:,}, d={D}")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** `Flat` y `IVFFlat` guardan los vectores completos (≈ $4d$ B). `HNSW` guarda **los vectores más el grafo** (cada nodo guarda sus vecinos): es *más grande* que `Flat`. `IVFPQ` guarda 16 bytes de código + id: ~10× menos. Elegir HNSW es pagar RAM por velocidad; elegir IVF-PQ, pagar recall por RAM.

## 5 · HNSW desde cero (mini)

**HNSW** (Malkov & Yashunin, 2016/2018) es un grafo de proximidad *jerárquico*:
- Cada vector recibe un **nivel máximo** aleatorio $\ell=\lfloor -\ln U\cdot m_L\rfloor$, con $m_L=1/\ln M$. Así $P(\ell\ge l)=M^{-l}$: cada capa tiene $M$ veces menos nodos que la inferior. Nº esperado de capas $\approx\ln N/\ln M$.
- **Búsqueda:** entra por la capa más alta con un nodo de entrada; en cada capa superior hace una **búsqueda voraz** (sigue al vecino más cercano a la query mientras mejore), baja a la capa siguiente con ese nodo y en la **capa 0** hace una búsqueda de haz con lista de tamaño $ef$.
- **Inserción:** igual que buscar para localizar los vecinos en cada capa $\le\ell$ (con `efConstruction`), conectar con $M$ vecinos elegidos por una **heurística** (acepta un candidato $c$ solo si está más cerca de la query que de cualquier vecino ya elegido: favorece vecinos en *direcciones distintas*) y recortar las listas de vecinos que excedan $M$ ($2M$ en la capa 0).
- **Parámetros:** `M` (conectividad: más memoria y recall), `efConstruction` (calidad del grafo, tiempo de construcción), `efSearch` (**el dial de latencia vs recall en consulta**).

Complejidad empírica: $O(\log N)$ saltos. Coste: RAM (vectores + grafo) y construcción lenta; no admite bien borrados.

In [ ]:
import heapq

class _HNSWBusqueda:
    '''Base del HNSW didáctico en NumPy puro: estructura, búsqueda por capa (haz de tamaño ef) y búsqueda jerárquica.
    Distancia L2: con vectores normalizados da el mismo ranking que el producto interno.'''
    def __init__(self, V, M=12, ef_construction=64, seed=SEED):
        self.X, self.n, self.M, self.M0, self.efc = V, len(V), M, 2 * M, ef_construction
        self.ml = 1 / math.log(M)
        self.rng = np.random.default_rng(seed)
        self.enlaces = []                       # enlaces[i][capa] = lista de vecinos
        self.entrada, self.capa_max, self.ndist = -1, -1, 0
    def _d(self, q, ids):
        self.ndist += len(ids)
        return ((self.X[ids] - q) ** 2).sum(1)
    def _busca_capa(self, q, eps, ef, capa, traza=None):
        vis = set(eps)
        d0 = self._d(q, list(eps))
        cand = list(zip(d0, eps)); heapq.heapify(cand)                    # min-heap de candidatos a expandir
        res = [(-d, e) for d, e in zip(d0, eps)]; heapq.heapify(res)       # max-heap de los ef mejores
        while cand:
            d, c = heapq.heappop(cand)
            if d > -res[0][0] and len(res) >= ef:
                break
            if traza is not None:
                traza.append(c)
            nb = [x for x in self.enlaces[c][capa] if x not in vis]
            if not nb:
                continue
            vis.update(nb)
            for dd, e in zip(self._d(q, nb), nb):
                if len(res) < ef or dd < -res[0][0]:
                    heapq.heappush(cand, (dd, e)); heapq.heappush(res, (-dd, e))
                    if len(res) > ef:
                        heapq.heappop(res)
        return sorted((-d, e) for d, e in res)
    def busca(self, q, k=10, ef=50, traza=False):
        ep, tr = [self.entrada], {}
        for capa in range(self.capa_max, 0, -1):
            tr[capa] = []
            ep = [self._busca_capa(q, ep, 1, capa, tr[capa])[0][1]]
        tr[0] = []
        res = self._busca_capa(q, ep, max(ef, k), 0, tr[0])[:k]
        ids = [e for _, e in res]
        return (ids, tr) if traza else ids

In [ ]:
class MiniHNSW(_HNSWBusqueda):
    '''Añade la INSERCIÓN: heurística de selección de vecinos, enlaces bidireccionales y recorte de grado.'''
    def _selecciona(self, cands, M):
        '''Heurística de selección: acepta c si está más cerca de la query que de cualquier vecino ya aceptado.'''
        out = []
        for dq, c in cands:
            if len(out) >= M:
                break
            if all(dq < ((self.X[c] - self.X[o]) ** 2).sum() for o in out):
                out.append(c)
        for dq, c in cands:                                               # completa hasta M si la heurística fue estricta
            if len(out) >= M:
                break
            if c not in out:
                out.append(c)
        return out
    def anade(self, i):
        q = self.X[i]
        nivel = int(-math.log(1 - self.rng.random()) * self.ml)
        self.enlaces.append([[] for _ in range(nivel + 1)])
        if self.entrada < 0:
            self.entrada, self.capa_max = i, nivel
            return
        ep = [self.entrada]
        for capa in range(self.capa_max, nivel, -1):                       # capas altas: búsqueda voraz (ef=1)
            ep = [self._busca_capa(q, ep, 1, capa)[0][1]]
        for capa in range(min(nivel, self.capa_max), -1, -1):
            W = self._busca_capa(q, ep, self.efc, capa)
            vecinos = self._selecciona(W, self.M)
            self.enlaces[i][capa] = list(vecinos)
            for e in vecinos:
                self.enlaces[e][capa].append(i)
                tope = self.M0 if capa == 0 else self.M
                if len(self.enlaces[e][capa]) > tope:                      # recorta con la misma heurística
                    ids = self.enlaces[e][capa]
                    self.enlaces[e][capa] = self._selecciona(sorted(zip(((self.X[ids] - self.X[e]) ** 2).sum(1), ids)), tope)
            ep = [e for _, e in W]
        if nivel > self.capa_max:
            self.entrada, self.capa_max = i, nivel
    def construye(self):
        for i in range(self.n):
            self.anade(i)
        return self

N_MINI = 2000 if FAST_DEV_RUN else 8000
Xm = X[:N_MINI]
GTm = exacto_topk(Q, Xm, 10)
t0 = time.perf_counter(); H = MiniHNSW(Xm, M=12, ef_construction=64).construye(); t_h = time.perf_counter() - t0
niveles = np.array([len(e) for e in H.enlaces])
print(f"mini-HNSW: N={N_MINI:,} construido en {t_h:.1f}s · capas={H.capa_max+1} · nodos por nivel máximo: {np.bincount(niveles)[1:].tolist()}")

filas = []
hf = faiss.IndexHNSWFlat(D, 12, faiss.METRIC_L2); hf.hnsw.efConstruction = 64; hf.add(Xm)
for ef in (10, 20, 40, 80, 160):
    H.ndist = 0
    res = [H.busca(q, 10, ef) for q in Q]
    hf.hnsw.efSearch = ef
    _, I = hf.search(Q, 10)
    filas.append(dict(ef=ef, recall_propio=recall_vs_gt(np.array(res), GTm), recall_faiss=recall_vs_gt(I, GTm), dist_por_query=H.ndist / len(Q),
                      fraccion_del_total=H.ndist / len(Q) / N_MINI))
tab_h = pd.DataFrame(filas)
display(tab_h.round(3))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
axs[0].plot(tab_h["ef"], tab_h["recall_propio"], "o-", label="mini-HNSW", color="#F28E2B")
axs[0].plot(tab_h["ef"], tab_h["recall_faiss"], "s--", label="FAISS IndexHNSWFlat", color="#1F6FEB")
axs[0].set_xscale("log", base=2); axs[0].set_xlabel("efSearch"); axs[0].set_ylabel("recall@10"); axs[0].set_title("Recall vs efSearch"); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3)
axs[1].plot(100 * tab_h["fraccion_del_total"], tab_h["recall_propio"], "o-", color="#F28E2B")
axs[1].set_xlabel("% de distancias calculadas respecto a fuerza bruta"); axs[1].set_ylabel("recall@10"); axs[1].set_title("Cuánto cuesta cada recall"); axs[1].grid(alpha=.3)
l = np.arange(0, H.capa_max + 1)
obs = np.array([(niveles > i).sum() for i in l])
axs[2].semilogy(l, obs, "o-", label="observado", color="#2CA02C"); axs[2].semilogy(l, N_MINI * 12.0 ** (-l), "k--", label="teoría: N·M^-l")
axs[2].set_xlabel("capa l"); axs[2].set_ylabel("nodos en capas ≥ l"); axs[2].set_title("Distribución de niveles"); axs[2].legend(fontsize=8); axs[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Tu mini-HNSW y el de FAISS tienen curvas parecidas: **`efSearch` es el dial** de recall/latencia. (2) En el segundo gráfico, alcanzar 0.95+ de recall requiere calcular solo una **fracción de las distancias** de la fuerza bruta (y esa fracción baja al crecer $N$ porque el coste es ≈ logarítmico). (3) El tercer gráfico confirma $P(\ell\ge l)=M^{-l}$: la capa $l$ tiene $\approx N M^{-l}$ nodos; por eso las capas altas son "autopistas" con pocos nodos.

### Ver las capas: HNSW en 2D
Construimos el mismo mini-HNSW sobre 150 puntos 2D y dibujamos las capas y el **recorrido** de una búsqueda: los nodos expandidos en cada capa (círculos rojos) entran por la capa alta y bajan hasta la zona de la query (estrella).

In [ ]:
P2 = (np.random.default_rng(3).normal(size=(150, 2)) * [1.0, 0.7]).astype(np.float32)
H2 = MiniHNSW(P2, M=4, ef_construction=16, seed=7).construye()
q2 = np.array([0.9, 0.5], dtype=np.float32)
ids2, traza2 = H2.busca(q2, k=3, ef=6, traza=True)
capas = H2.capa_max + 1
fig, axs = plt.subplots(1, capas, figsize=(4.2 * capas, 4), sharex=True, sharey=True)
axs = np.atleast_1d(axs)
for capa in range(capas):
    ax = axs[capas - 1 - capa]
    en_capa = [i for i, e in enumerate(H2.enlaces) if len(e) > capa]
    ax.scatter(P2[:, 0], P2[:, 1], s=6, c="#D1D5DB")
    for i in en_capa:
        for j in H2.enlaces[i][capa]:
            ax.plot(*P2[[i, j]].T, c="#1F6FEB", lw=0.5, alpha=.5)
    ax.scatter(P2[en_capa, 0], P2[en_capa, 1], s=22, c="#1F6FEB")
    vis = traza2.get(capa, [])
    ax.scatter(P2[vis, 0], P2[vis, 1], s=55, facecolors="none", edgecolors="#D62728", lw=1.6)
    ax.scatter(*q2, marker="*", s=200, c="#F28E2B", zorder=5)
    ax.set_title(f"capa {capa}" + (" (arriba)" if capa == capas - 1 else " (base)" if capa == 0 else ""), fontsize=10)
plt.suptitle("HNSW en 2D: nodos (azul), nodos expandidos (círculo rojo), query (estrella)", fontsize=10)
plt.tight_layout(); plt.show()
print({f"capa {c}": len(v) for c, v in sorted(traza2.items(), reverse=True)}, "nodos expandidos")

> 👀 **Qué debes observar.** En las capas altas hay pocos nodos y aristas **largas**: la búsqueda se acerca a la zona de la query en unos pocos saltos. En la capa 0 (todos los nodos, aristas cortas) solo se expande un puñado de nodos alrededor de la estrella. Esa es la intuición de $O(\log N)$: la jerarquía hace de **índice de saltos** sobre un grafo local.

> ### 🎬 GEMINI-VIDEO · ID: `M14-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Recorrido de HNSW por capas**
>
> **Qué muestra:** Animación de una búsqueda en HNSW: entrada por la capa superior, descenso voraz capa a capa y búsqueda con lista de candidatos en la capa 0.
> **Formato:** MP4 16:9, 1080p | Duración: 25-30 s | **Texto alternativo:** Animación de una búsqueda HNSW que desciende por tres capas hasta los vecinos más cercanos
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo 16:9, fondo blanco, estilo diagrama técnico limpio. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280; tipografía sans-serif; ritmo calmado; SIN voz en off ni música; solo texto en pantalla en español (términos técnicos en inglés).
> 
> Composición: tres paneles horizontales apilados que representan las capas 2 (arriba), 1 (centro) y 0 (abajo) de un grafo HNSW. En la capa 2 hay 4 nodos grises con 4 aristas largas; en la capa 1 hay 10 nodos; en la capa 0 hay 40 nodos con muchas aristas cortas. Los nodos de las capas altas están conectados con líneas verticales punteadas a los mismos nodos en la capa de abajo. Una estrella naranja (la consulta) está en la zona derecha de la capa 0.
> 
> Escena 1 (0–4 s): título "HNSW: búsqueda por capas". Aparecen los tres paneles con sus nodos.
> Escena 2 (4–10 s): el punto de entrada (nodo azul oscuro, arriba a la izquierda de la capa 2) se ilumina con la etiqueta "punto de entrada". Se comparan sus vecinos y el algoritmo salta al vecino más cercano a la estrella, resaltando la arista en verde. Texto: "capa 2: búsqueda voraz (ef = 1)".
> Escena 3 (10–15 s): una flecha vertical punteada baja el nodo actual a la capa 1; se repite el salto voraz hasta no poder mejorar. Texto: "capa 1: seguir mejorando".
> Escena 4 (15–24 s): baja a la capa 0; ahora se resalta una lista de candidatos (halo naranja alrededor de 6 nodos cercanos a la estrella) que se va actualizando: entran nodos nuevos y salen los más lejanos. Texto: "capa 0: lista de candidatos de tamaño ef".
> Escena 5 (24–28 s): los 3 nodos finales más cercanos a la estrella se marcan en verde con la etiqueta "top-k". Texto final: "Pocos saltos arriba + exploración local abajo ≈ O(log N)".
> 
> Evita: voz, música, nodos que aparecen y desaparecen sin motivo, texto ilegible, decoración, cambios bruscos de cámara.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M14-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M14-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> capa 2:  (E)────────o        o──────o         E = punto de entrada
>           ↓ salto voraz hacia la consulta ★
> capa 1:  o──o──(o)──o──o──o                     ↓ baja con el mejor nodo
> capa 0:  o-o-o-o-(o)-o-o-o-o-★   lista de candidatos (ef) -> top-k
> ```

> 👀 **Qué debes observar.** El tiempo se reparte así: **pocos saltos en las capas altas** (guía gruesa), **mucha exploración en la capa 0** (precisión). `ef` controla cuántos candidatos se mantienen en esa exploración: más `ef` = más recall = más distancias.

## 6 · Vamana y DiskANN: grafos pensados para el SSD

HNSW exige **todo en RAM**. Para $10^9$ vectores eso son terabytes. **DiskANN** (Subramanya et al., NeurIPS 2019) resuelve el problema de dos maneras:

1. **Un grafo distinto, Vamana**, con grado acotado $R$ y **pocos saltos** (cada salto es una lectura aleatoria de SSD, ≈ 100 µs en NVMe: el número de saltos *es* la latencia).
2. **Arquitectura híbrida:** en RAM solo los **códigos PQ** de todos los vectores (para guiar la búsqueda con distancias aproximadas); en SSD, para cada nodo, su **vector completo y su lista de vecinos** juntos en un sector. La búsqueda por haz (*beam search*) lee `W` sectores por ronda y re-ordena con los vectores completos recién leídos.

**Construcción de Vamana.** Parte de un grafo aleatorio $R$-regular y de un nodo de entrada (*medoide*). Para cada punto $p$ (en orden aleatorio): (1) ejecuta `GreedySearch` desde el medoide hacia $p$ y guarda el conjunto $V$ de nodos **visitados**; (2) fija los vecinos de $p$ con **RobustPrune**$(p,V,\alpha,R)$; (3) añade las aristas inversas y recorta con RobustPrune los nodos que se pasan de grado $R$. Se hacen **dos pasadas**, primero con $\alpha=1$ y luego con $\alpha>1$ (p. ej. 1.2).

**RobustPrune.** Toma el candidato $p^\*$ más cercano a $p$, lo añade a los vecinos y **elimina** todos los $p'$ que quedan "cubiertos" por él: $\alpha\cdot d(p^\*,p')\le d(p,p')$. Con $\alpha=1$ es la poda tipo HNSW/RNG (solo aristas "útiles"); con $\alpha>1$ se conservan **aristas más largas**, lo que reduce el diámetro del grafo y por tanto el nº de saltos (a costa de más aristas por nodo).

Otras piezas: **FreshDiskANN** (Singh et al., 2021) soporta inserciones y borrados en streaming; **Filtered-DiskANN** añade filtros por etiqueta en la construcción del grafo.

In [ ]:
class Vamana:
    '''Vamana didáctico (distancia coseno sobre vectores normalizados).'''
    def __init__(self, V, R=16, L=48, alpha=1.2, seed=SEED):
        self.X, self.n, self.R, self.ndist = V, len(V), R, 0
        rng = np.random.default_rng(seed)
        self.g = [list(rng.choice(self.n, R, replace=False)) for _ in range(self.n)]     # grafo aleatorio R-regular
        self.s = int(np.argmin(((V - V.mean(0)) ** 2).sum(1)))                          # medoide aproximado = punto más cercano a la media
        for a in (1.0, alpha):                                                           # dos pasadas: alpha=1 y alpha>1
            for p in rng.permutation(self.n):
                visitados = self.voraz(V[p], L)[1]
                visitados.discard(int(p))
                self.poda(int(p), visitados, a)
                for j in self.g[p]:
                    if p not in self.g[j]:
                        self.g[j].append(int(p))
                        if len(self.g[j]) > R:
                            self.poda(j, set(self.g[j]), a)
        self.ndist = 0
    def voraz(self, q, L):
        '''GreedySearch: devuelve (candidatos ordenados, conjunto de nodos expandidos = lecturas de página).'''
        vis, cand, expandidos = {self.s}, {self.s: 1 - self.X[self.s] @ q}, set()
        self.ndist += 1
        while True:
            pend = [(d, c) for c, d in cand.items() if c not in expandidos]
            if not pend:
                break
            _, c = min(pend)
            expandidos.add(c)
            nb = [x for x in self.g[c] if x not in vis]
            if nb:
                vis.update(nb); self.ndist += len(nb)
                for x, d in zip(nb, 1 - self.X[nb] @ q):
                    cand[x] = d
            if len(cand) > L:
                cand = dict(sorted(cand.items(), key=lambda kv: kv[1])[:L])
        return sorted(cand, key=cand.get), expandidos
    def poda(self, p, V, alpha):
        '''RobustPrune: añade el más cercano y elimina a los que quedan cubiertos (alpha * d(p*, p') <= d(p, p')).'''
        V = list(V)
        d = 1 - self.X[V] @ self.X[p]
        orden = np.argsort(d); V = [V[i] for i in orden]; d = d[orden]
        salida, vivo = [], np.ones(len(V), bool)
        for i in range(len(V)):
            if not vivo[i]:
                continue
            salida.append(V[i])
            if len(salida) >= self.R:
                break
            dd = 1 - self.X[V[i + 1:]] @ self.X[V[i]]
            vivo[i + 1:] &= ~(alpha * dd <= d[i + 1:])
        self.g[p] = salida
    def busca(self, q, k=10, L=30):
        res, lecturas = self.voraz(q, L)
        return res[:k], len(lecturas)

N_V = 1500 if FAST_DEV_RUN else 5000
Xv, GTv = X[:N_V], exacto_topk(Q, X[:N_V], 10)
filas = []
for alpha in (1.0, 1.2):
    t0 = time.perf_counter(); V_ = Vamana(Xv, R=16, L=48, alpha=alpha); t_b = time.perf_counter() - t0
    grado = np.mean([len(v) for v in V_.g])
    for L in (10, 20, 40, 80):
        res = [V_.busca(q, 10, L) for q in Q]
        filas.append(dict(alpha=alpha, L=L, recall=recall_vs_gt(np.array([r[0] for r in res]), GTv), lecturas_por_query=np.mean([r[1] for r in res]), grado_medio=grado, seg_construir=t_b))
# referencia: nodos expandidos por el mini-HNSW (capas altas + capa 0)
Hv = MiniHNSW(Xv, M=12, ef_construction=64).construye()
for ef in (10, 20, 40, 80):
    out = [Hv.busca(q, 10, ef, traza=True) for q in Q]
    filas.append(dict(alpha=np.nan, L=ef, recall=recall_vs_gt(np.array([o[0] for o in out]), GTv),
                      lecturas_por_query=np.mean([sum(len(t) for t in o[1].values()) for o in out]), grado_medio=np.mean([len(e[0]) for e in Hv.enlaces]), seg_construir=np.nan))
tab_v = pd.DataFrame(filas)
tab_v["algoritmo"] = np.where(tab_v["alpha"].isna(), "mini-HNSW", "Vamana α=" + tab_v["alpha"].astype(str))
display(tab_v.round(3))

In [ ]:
T_LECTURA_MS = 0.1      # SUPUESTO: ≈ 100 µs por lectura aleatoria de 4 KB en NVMe (ajústalo a tu disco); lecturas secuenciales por ronda de beam W=1
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
for alg, g in tab_v.groupby("algoritmo"):
    axs[0].plot(g["lecturas_por_query"], g["recall"], "o-", label=alg)
axs[0].set_xlabel("nodos expandidos por query (≈ lecturas de SSD)"); axs[0].set_ylabel("recall@10"); axs[0].set_title("Recall vs lecturas de página"); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3)
for alg, g in tab_v.groupby("algoritmo"):
    axs[1].plot(g["lecturas_por_query"] * T_LECTURA_MS, g["recall"], "o-", label=alg)
axs[1].set_xlabel(f"latencia estimada de I/O (ms) con {T_LECTURA_MS*1e3:.0f} µs por lectura"); axs[1].set_ylabel("recall@10"); axs[1].set_title("Si el grafo viviera en SSD (estimación)"); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Vamana alcanza recalls comparables al mini-HNSW con un número de **nodos expandidos** del mismo orden: el grafo es de calidad similar. (2) En esta escala diminuta (1 500 nodos) la diferencia entre $\alpha=1$ y $\alpha=1.2$ es pequeña; en el paper el efecto de $\alpha>1$ se nota en colecciones grandes, donde acorta el recorrido. (3) A la derecha, el eje es **latencia de I/O estimada** con un supuesto explícito de 100 µs por lectura: es lo que realmente paga DiskANN, y por eso minimiza lecturas, no distancias. (4) Un diseño con PQ en RAM evita leer los vecinos "malos": se usa la distancia aproximada para decidir qué sector leer.

> ### 🎨 GEMINI-IMAGEN · ID: `M14-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Arquitectura de DiskANN: grafo en SSD y PQ en RAM**
>
> **Qué muestra:** Qué vive en RAM (códigos PQ), qué vive en SSD (vector completo + vecinos de cada nodo) y cómo una búsqueda por haz lee pocos sectores.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Esquema con los códigos PQ en RAM, los sectores con vector y vecinos en SSD, y el flujo de búsqueda por haz
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif. Sin decoración.
> 
> Dos zonas verticales separadas por una línea punteada.
> IZQUIERDA, azul, título "RAM (pequeña)": una tabla con 8 filas "id 0 … id 7" y una columna "código PQ (m bytes)" con ejemplos "[17 203 5 88]" en las dos primeras filas y puntos suspensivos. Debajo, la nota: "distancias aproximadas para decidir qué sector leer".
> DERECHA, naranja, título "SSD (grande)": una fila de 6 rectángulos alargados llamados "sector 4 KB", cada uno con dos partes: "vector completo" (barra larga) y "vecinos: [a, b, c, d]" (4 casillas).
> Entre ambas zonas, flechas: 1) desde un cuadro verde "consulta q" hacia la tabla PQ, con la etiqueta "1. puntuar vecinos con PQ"; 2) desde la tabla PQ hacia 2 sectores del SSD con la etiqueta "2. leer solo los W mejores sectores (beam search)"; 3) desde esos sectores hacia una caja verde "3. re-ordenar con vectores completos" y luego "top-k".
> Pie de figura: "Latencia ≈ nº de lecturas de SSD · RAM ≈ N × m bytes".
> 
> Evita: texto ilegible, más de 6 sectores, números distintos de los indicados, logos, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M14-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Esquema con los códigos PQ en RAM, los sectores con vector y vecinos en SSD, y el flujo de búsqueda por haz](assets/M14-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> RAM:  id0 [17 203 5 88]            SSD: [sector: vector completo | vecinos a b c d] x N
>       id1 [ 9  44 120 3]    ←→      1) q puntúa vecinos con PQ  2) lee W sectores  3) re-ordena con vectores completos
> RAM ≈ N·m bytes · latencia ≈ nº de lecturas de SSD
> ```

> 👀 **Qué debes observar.** Dos cuellos de botella distintos: la **RAM** escala con $N\cdot m$ (bytes de PQ), la **latencia** con el número de lecturas. DiskANN permite que $N$ crezca 10–100× más que lo que cabe en RAM manteniendo latencias de milisegundos.

## 7 · ScaNN: cuantización *anisótropa*

La PQ minimiza el error de reconstrucción $\|x-\tilde x\|^2$. Pero en búsqueda por producto interno (MIPS) lo que importa es el error del **score** $\langle q,x\rangle$, y ese error no afecta por igual a todos los pares: para los vecinos que **sí** serán top-$k$ (los de score alto), la query está casi alineada con $x$. Descomponiendo el residuo $r=x-\tilde x$ en la componente **paralela** $r_\parallel$ (en la dirección de $x$) y la **ortogonal** $r_\perp$, el error de score proviene sobre todo de $r_\parallel$. ScaNN (Guo et al., ICML 2020) minimiza una pérdida **ponderada**
$$\ell(x,\tilde x)=h_\parallel\,\|r_\parallel\|^2+h_\perp\,\|r_\perp\|^2,\qquad h_\parallel>h_\perp,$$
con $h_\parallel/h_\perp$ derivado del umbral de score que se quiere preservar y de la dimensión. Resultado: códigos que **sacrifican error ortogonal** (que casi no altera el ranking) para **proteger los scores altos**. Es la base de la búsqueda vectorial de Google (`scann`: árbol/particionado + *asymmetric hashing* anisótropo + re-ranking).

In [ ]:
# Ejemplo numérico mínimo: dos palabras de código candidatas para x = (1, 0) y consultas alineadas con x
x = np.array([1.0, 0.0])
c1, c2 = np.array([0.8, 0.0]), np.array([1.0, 0.25])          # c1 = error PARALELO 0.2 · c2 = error ORTOGONAL 0.25
def error(c):
    r = x - c; r_par = (r @ x) / (x @ x) * x; return r_par, r - r_par
for nombre, c in (("c1 (error paralelo)", c1), ("c2 (error ortogonal)", c2)):
    rp, ro = error(c)
    mse = float((rp ** 2).sum() + (ro ** 2).sum())
    aniso = 4 * float((rp ** 2).sum()) + float((ro ** 2).sum())                  # h_par/h_perp = 4 (ejemplo)
    print(f"{nombre}: MSE = {mse:.4f} · pérdida anisótropa (η=4) = {aniso:.4f} · score con q=(1,0): <q,c> = {c[0]:.2f} (verdadero: 1.00)")
elige_mse = "c1" if ((error(c1)[0] ** 2).sum() + (error(c1)[1] ** 2).sum()) < ((error(c2)[0] ** 2).sum() + (error(c2)[1] ** 2).sum()) else "c2"
elige_aniso = "c1" if (4 * (error(c1)[0] ** 2).sum() + (error(c1)[1] ** 2).sum()) < (4 * (error(c2)[0] ** 2).sum() + (error(c2)[1] ** 2).sum()) else "c2"
print(f"PQ clásica (MSE) elige {elige_mse}; ScaNN (η=4) elige {elige_aniso}")
fig, ax = plt.subplots(figsize=(4.6, 3.8))
ax.annotate("", xy=x, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#1F6FEB", lw=2)); ax.text(0.96, -0.08, "x", color="#1F6FEB")
for c, col, et in ((c1, "#F28E2B", "c1"), (c2, "#2CA02C", "c2")):
    ax.scatter(*c, c=col, s=60); ax.text(c[0] + 0.02, c[1] + 0.02, et, color=col)
    ax.plot([x[0], c[0]], [x[1], c[1]], ls="--", c=col, lw=1)
ax.set_xlim(0.6, 1.15); ax.set_ylim(-0.15, 0.4); ax.set_aspect("equal"); ax.grid(alpha=.3); ax.set_title("Residuo paralelo (c1) vs ortogonal (c2)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La PQ clásica prefiere $c_1$ (menor error cuadrático total) pero $c_1$ **subestima el score** de las consultas alineadas con $x$ (0.80 en vez de 1.00); la pérdida anisótropa prefiere $c_2$, que conserva el score exacto aunque su error total sea mayor. En la práctica el efecto es sutil y se mide con recall del top-$k$ de MIPS; ScaNN lo combina con particionado y re-ranking.

La librería oficial se instala con `pip install scann` (solo Linux x86). La celda siguiente la usa si está disponible (si no, la omite):

In [ ]:
try:
    import scann
    buscador = (scann.scann_ops_pybind.builder(X, 10, "dot_product")
                .tree(num_leaves=NLIST, num_leaves_to_search=16, training_sample_size=min(N, 10000))
                .score_ah(2, anisotropic_quantization_threshold=0.2)
                .reorder(100).build())
    ids, _ = buscador.search_batched(Q)
    print("ScaNN recall@10:", round(recall_vs_gt(np.asarray(ids), GT), 3))
except Exception as e:
    print("scann no disponible en este entorno:", type(e).__name__, "· (opcional: pip install scann en Linux x86)")

## 8 · Cuantización binaria con garantías: RaBitQ

El módulo 12 mostró la binaria "ingenua" ($\text{sign}(x)$). **RaBitQ** (Gao & Long, SIGMOD 2024) la convierte en un método con **estimador insesgado y cota de error**: $D$ bits por vector y un error $O(1/\sqrt D)$ con alta probabilidad.

**Construcción** (por celda IVF con centroide $c$; aquí un único centroide global = la media):
1. Residuo normalizado: $o=\dfrac{x-c}{\|x-c\|}$ (guardando $\|x-c\|$ y el escalar $\langle\bar o,o\rangle$ de abajo).
2. Se aplica una **rotación ortogonal aleatoria** $P$ y se toma el vértice del hipercubo más cercano: $b=\mathbb{1}[P^\top o>0]$, $\bar o=P\,\dfrac{2b-1}{\sqrt D}$ (un vector unitario). Solo se guarda $b$ ($D$ bits).
3. **Estimador del producto interno** con la query residual normalizada $q_r$:
$$\langle o,q_r\rangle\;\approx\;\frac{\langle\bar o,q_r\rangle}{\langle\bar o,o\rangle}$$
El numerador se calcula con operaciones de bits sobre $b$ (la query rotada); el denominador es un **factor de corrección** precomputado por vector que hace el estimador **insesgado** (sin él, la binaria "ingenua" subestima sistemáticamente los productos internos altos). La distancia se reconstruye con $\|x-q\|^2=\|x-c\|^2+\|q-c\|^2-2\|x-c\|\|q-c\|\langle o,q_r\rangle$.

**Extended RaBitQ** (2409.09913) generaliza a más de 1 bit por dimensión. En la práctica los sistemas usan el estimador para filtrar y **re-puntúan** con vectores completos (como en el rescoring del módulo 12).

In [ ]:
def rabitq_ajusta(V, seed=SEED):
    D_ = V.shape[1]
    c = V.mean(0)
    P, _ = np.linalg.qr(np.random.default_rng(seed).normal(size=(D_, D_)))             # rotación ortogonal aleatoria
    R_ = V - c; nr = np.linalg.norm(R_, axis=1); o = R_ / nr[:, None]
    z = o @ P                                                                          # P^T o
    bits = z > 0
    ip = (z * (2 * bits - 1)).sum(1) / math.sqrt(D_)                                    # <obar, o>: factor de corrección
    return dict(c=c, P=P, bits=bits, nr=nr, ip=ip, D=D_, o=o)

def rabitq_ip(st, q):
    qr = q - st["c"]; nq = np.linalg.norm(qr)
    zq = (qr / nq) @ st["P"]
    return ((2 * st["bits"] - 1) @ zq) / math.sqrt(st["D"]) / st["ip"], nq            # estimador insesgado de <o, q_r>

def sign_ingenuo(st, q):
    qr = (q - st["c"]) / np.linalg.norm(q - st["c"])
    return (np.sign(st["o"]) / math.sqrt(st["D"])) @ qr                               # sign(o)/sqrt(D) sin rotación ni corrección

rb = rabitq_ajusta(X)
ratios_rq, ratios_ing, errores = [], [], []
for q in Q[:100]:
    est, nq = rabitq_ip(rb, q)
    real = rb["o"] @ ((q - rb["c"]) / nq)
    top = np.argsort(-real)[:20]                                                       # los vecinos que importan: los de producto interno alto
    ratios_rq.append(np.mean(est[top] / real[top])); ratios_ing.append(np.mean(sign_ingenuo(rb, q)[top] / real[top]))
    errores.append(np.std(est - real))
print(f"estimado/real en los 20 mejores vecinos: RaBitQ = {np.mean(ratios_rq):.3f} · sign ingenuo = {np.mean(ratios_ing):.3f}  (1.0 = insesgado)")

# Error vs dimensión: debe escalar como 1/sqrt(D)
dims = [32, 64, 128, 256, 512]
err_d = []
for d_ in dims:
    Xd, Qd = genera_vectores(2000, d_, nq=20)
    s = rabitq_ajusta(Xd); e = []
    for q in Qd:
        est, nq = rabitq_ip(s, q); e.append(np.std(est - s["o"] @ ((q - s["c"]) / nq)))
    err_d.append(np.mean(e))
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
q = Q[0]; est, nq = rabitq_ip(rb, q); real = rb["o"] @ ((q - rb["c"]) / nq)
axs[0].scatter(real[:1500], est[:1500], s=4, alpha=.5, c="#2CA02C", label="RaBitQ"); axs[0].scatter(real[:1500], sign_ingenuo(rb, q)[:1500], s=4, alpha=.4, c="#F28E2B", label="sign ingenuo")
lim = [real.min(), real.max()]; axs[0].plot(lim, lim, "k--", lw=.8); axs[0].set_xlabel("<o, q> real"); axs[0].set_ylabel("estimado"); axs[0].set_title("Estimador vs valor real (una query)"); axs[0].legend(fontsize=8)
axs[1].bar(["RaBitQ", "sign ingenuo"], [np.mean(ratios_rq), np.mean(ratios_ing)], color=["#2CA02C", "#F28E2B"]); axs[1].axhline(1, color="k", ls="--"); axs[1].set_ylabel("estimado / real (top-20)"); axs[1].set_title("Sesgo en los vecinos que importan")
axs[2].loglog(dims, err_d, "o-", color="#1F6FEB", label="error medido"); axs[2].loglog(dims, err_d[0] * np.sqrt(dims[0] / np.array(dims)), "k--", label="∝ 1/√D")
axs[2].set_xlabel("dimensión D"); axs[2].set_ylabel("desviación del error"); axs[2].set_title("El error baja como 1/√D"); axs[2].legend(fontsize=8); axs[2].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) A la izquierda, RaBitQ se agrupa en torno a la diagonal; el *sign* ingenuo queda **por debajo** (escala comprimida). (2) En el centro, sobre los vecinos que importan, el ingenuo estima ≈ 0.7–0.8 del valor real (sesgo) mientras RaBitQ ≈ 1.0 (insesgado): eso es lo que arregla la división por $\langle\bar o,o\rangle$. (3) A la derecha, la desviación del error cae como $1/\sqrt D$ (recta de pendiente −1/2 en log-log): **más dimensiones = mejor estimación con 1 bit por dimensión**, lo contrario de lo que pasa en PQ. Esa propiedad hace atractiva la binaria para embeddings de 768–3072 dimensiones.

> 🏭 **Adopción (fuentes):** Elastic describe su *Better Binary Quantization* (BBQ, Elasticsearch 8.16) como construido sobre ideas de RaBitQ, con diferencias (un único centroide, sin rotación aleatoria del libro de códigos); Milvus informa soporte de RaBitQ desde la 2.6 y haber contribuido una versión optimizada a FAISS. Son afirmaciones de cada proyecto: verifica la versión que uses.

## 9 · Búsqueda filtrada: el problema abierto de la producción

Políglota necesita consultas como «solo documentación en español de la versión 3.x». Tres estrategias:

| Estrategia | Cómo | Problema |
|---|---|---|
| **Post-filtro** | buscar top-$k'$ ($k'\gg k$) y filtrar | si el filtro es muy selectivo, no queda nada en el top-$k'$ |
| **Pre-filtro** | construir la lista permitida y hacer **fuerza bruta** sobre ella | exacto, pero el coste crece con $s\cdot N$ ($s$ = selectividad) |
| **Filtro dentro del grafo** | ignorar al expandir los nodos no permitidos | el subgrafo permitido puede quedar **desconectado**: recall cae con $s$ pequeño |

ACORN (Patel et al., 2024) y Filtered-DiskANN atacan el problema modificando la construcción del grafo; los motores suelen **elegir la estrategia según la selectividad** estimada.

In [ ]:
rng = np.random.default_rng(SEED)
atributo = rng.random(N)                                   # atributo uniforme: selectividad = fracción permitida
hnsw = faiss.IndexHNSWFlat(D, 16, faiss.METRIC_INNER_PRODUCT); hnsw.hnsw.efConstruction = 100; hnsw.add(X)
EF, SOBREMUESTREO = 64, 10
filas = []
for sel in (0.5, 0.2, 0.05, 0.01, 0.002):
    ok = np.where(atributo < sel)[0]
    gt_f = ok[exacto_topk(Q, X[ok], 10)] if len(ok) > 10 else None
    if gt_f is None:
        continue
    # (a) post-filtro: top-(10*sobremuestreo) y quedarse con los permitidos
    hnsw.hnsw.efSearch = max(EF, 10 * SOBREMUESTREO)
    _, I = hnsw.search(Q, 10 * SOBREMUESTREO)
    permitido = set(ok.tolist())
    post = [[x for x in fila if x in permitido][:10] for fila in I]
    # (b) filtro dentro del grafo
    p = faiss.SearchParametersHNSW(); p.efSearch = EF; p.sel = faiss.IDSelectorBatch(ok)
    t0 = time.perf_counter(); _, I2 = hnsw.search(Q, 10, params=p); t_in = (time.perf_counter() - t0) / len(Q) * 1e3
    # (c) pre-filtro: fuerza bruta sobre los permitidos
    sub = np.ascontiguousarray(X[ok])
    t0 = time.perf_counter(); [ (sub @ q) for q in Q ]; t_pre = (time.perf_counter() - t0) / len(Q) * 1e3
    filas.append(dict(selectividad=sel, n_permitidos=len(ok), recall_post=recall_vs_gt(np.array([r + [-1] * (10 - len(r)) for r in post]), gt_f),
                      recall_en_grafo=recall_vs_gt(I2, gt_f), recall_pre=1.0, ms_en_grafo=t_in, ms_prefiltro=t_pre))
tab_f = pd.DataFrame(filas)
display(tab_f.round(3))
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].semilogx(tab_f["selectividad"], tab_f["recall_post"], "o-", label=f"post-filtro (x{SOBREMUESTREO})", color="#F28E2B")
axs[0].semilogx(tab_f["selectividad"], tab_f["recall_en_grafo"], "s-", label="filtro dentro del grafo", color="#1F6FEB")
axs[0].semilogx(tab_f["selectividad"], tab_f["recall_pre"], "^-", label="pre-filtro (exacto)", color="#2CA02C")
axs[0].set_xlabel("selectividad (fracción permitida)"); axs[0].set_ylabel("recall@10 vs exacto filtrado"); axs[0].set_title("El recall se rompe con filtros selectivos"); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3, which="both")
axs[1].loglog(tab_f["selectividad"], tab_f["ms_prefiltro"], "^-", color="#2CA02C", label="pre-filtro (∝ s·N)")
axs[1].loglog(tab_f["selectividad"], tab_f["ms_en_grafo"], "s-", color="#1F6FEB", label="en el grafo")
axs[1].set_xlabel("selectividad"); axs[1].set_ylabel("ms por query"); axs[1].set_title("Latencia"); axs[1].legend(fontsize=8); axs[1].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Con filtros poco selectivos (50 %) todo funciona. (2) Al bajar la selectividad, el **post-filtro** colapsa (no hay suficientes permitidos en el top-100) y el **filtro dentro del grafo** también pierde recall (el subgrafo permitido se desconecta). (3) El **pre-filtro** es exacto y su coste *baja* con la selectividad (menos vectores que escanear): para filtros muy selectivos es **la mejor opción**. Regla de ingeniería: estimar la selectividad y elegir estrategia (pre-filtro si $s\cdot N$ es pequeño; filtro en grafo si es moderado; post-filtro solo si $s$ es alto).

## 10 · FAISS en uso real: barrer configuraciones y elegir con un SLA

El flujo de trabajo de un ingeniero de search: (1) fijar el **SLA** (p. ej. recall@10 ≥ 0.95, p99 ≤ 5 ms), (2) barrer familias y parámetros, (3) quedarse con la frontera de Pareto y (4) elegir el punto que cumple el SLA con **menos memoria**. `index_factory` describe una configuración en una cadena (`"IVF141,PQ16"`, `"HNSW16"`, `"IVF141,Flat"`).

In [ ]:
def barre(familia, construir, parametros, aplica):
    ix = construir()
    if not ix.is_trained:
        ix.train(X)
    t0 = time.perf_counter(); ix.add(X); t_b = time.perf_counter() - t0
    mb = tam_indice_MB(ix); out = []
    for pm in parametros:
        aplica(ix, pm)
        _, I = ix.search(Q, 10)
        p50, p99, qps = mide_latencia(lambda q: ix.search(q[None], 10), Q[:100])
        out.append(dict(familia=familia, param=pm, recall=recall_vs_gt(I, GT), p50_ms=p50, p99_ms=p99, QPS=qps, MB=mb, seg_construir=t_b))
    return out

_MANTENER_VIVOS = []

def mk_ivfpq_refine():
    base = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, NLIST, 16, 8, faiss.METRIC_INNER_PRODUCT)
    ix = faiss.IndexRefineFlat(base)
    _MANTENER_VIVOS.append(base)                               # FAISS no es dueño del índice base: hay que mantener la referencia
    return ix

filas = []
filas += barre("IVF-Flat", lambda: faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, NLIST, faiss.METRIC_INNER_PRODUCT), [1, 2, 4, 8, 16, 32],
               lambda ix, p: setattr(ix, "nprobe", p))
filas += barre("IVF-PQ16", lambda: faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, NLIST, 16, 8, faiss.METRIC_INNER_PRODUCT), [4, 8, 16, 32, 64],
               lambda ix, p: setattr(ix, "nprobe", p))
filas += barre("IVF-PQ16 + refine x10", mk_ivfpq_refine, [4, 8, 16, 32],
               lambda ix, p: (setattr(faiss.extract_index_ivf(ix.base_index), "nprobe", p), setattr(ix, "k_factor", 10)))
filas += barre("HNSW16", lambda: (lambda h: (setattr(h.hnsw, "efConstruction", 100), h)[1])(faiss.IndexHNSWFlat(D, 16, faiss.METRIC_INNER_PRODUCT)),
               [8, 16, 32, 64, 128, 256], lambda ix, p: setattr(ix.hnsw, "efSearch", p))
barrido = pd.DataFrame(filas)
display(barrido.round(3).head(8))

In [ ]:
colores = {"IVF-Flat": "#1F6FEB", "IVF-PQ16": "#2CA02C", "IVF-PQ16 + refine x10": "#9467BD", "HNSW16": "#F28E2B"}
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for fam, g in barrido.groupby("familia"):
    axs[0].plot(g["p50_ms"], g["recall"], "o-", label=f"{fam} ({g['MB'].iloc[0]:.1f} MB)", color=colores[fam])
axs[0].axhline(0.95, color="k", ls=":"); axs[0].text(axs[0].get_xlim()[0], 0.955, " SLA recall ≥ 0.95", fontsize=8)
axs[0].set_xscale("log"); axs[0].set_xlabel("latencia p50 por query (ms, escala log)"); axs[0].set_ylabel("recall@10"); axs[0].set_title("Recall vs latencia (leyenda: memoria del índice)")
axs[0].legend(fontsize=8); axs[0].grid(alpha=.3, which="both")
ok = barrido[barrido["recall"] >= 0.95]
for fam, g in barrido.groupby("familia"):
    axs[1].scatter(g["MB"], g["p99_ms"], s=30 + 300 * g["recall"] ** 8, alpha=.6, color=colores[fam], label=fam)
axs[1].set_xscale("log"); axs[1].set_yscale("log"); axs[1].set_xlabel("memoria del índice (MB)"); axs[1].set_ylabel("p99 (ms)"); axs[1].set_title("Memoria vs p99 (tamaño = recall)"); axs[1].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()
if len(ok):
    mejor = ok.sort_values(["MB", "p99_ms"]).iloc[0]
    print(f"Con SLA recall≥0.95, la config de menor memoria es {mejor['familia']} (param={mejor['param']}): recall={mejor['recall']:.3f}, p99={mejor['p99_ms']:.3f} ms, {mejor['MB']:.1f} MB")
else:
    print("Ninguna configuración llega a recall 0.95 en este barrido: amplía los parámetros.")

> 👀 **Qué debes observar.** (1) Cada familia traza **su propia curva**: IVF-PQ ocupa ≈ 10× menos memoria pero **se estanca en un recall bajo** (la compresión introduce un error que más `nprobe` no arregla); IVF-PQ + *refine* devuelve el recall perdido a cambio de guardar los vectores completos (que pueden vivir en disco). (2) **Sorpresa honesta:** en esta escala diminuta ($N=20\,000$) y con estos datos poco estructurados, IVF-Flat suele igualar o superar a HNSW en latencia a recall ≈ 0.95. Los benchmarks públicos (*ann-benchmarks*, Aumüller et al.) muestran que los grafos tienden a dominar a recall alto en colecciones **grandes**, porque su coste crece ≈ logarítmicamente con $N$ y el de IVF ≈ con $\sqrt N$: **no extrapoles de $N$ pequeño; mide a tu escala**. (3) Con el mismo SLA de recall, la configuración ganadora depende de qué restricción pesa más: latencia, memoria o tiempo de construcción (HNSW tarda órdenes de magnitud más en construirse que IVF aquí). Tamaño de la burbuja del segundo gráfico: recall.

### La dificultad la pone el dato
Los parámetros óptimos dependen de la **estructura de los vectores**. Mismas configuraciones, tres conjuntos con distinta anisotropía (`decay`): casi isótropo (difícil), tal como los usamos, y muy anisótropo (los embeddings reales suelen tener espectros fuertemente decrecientes).

In [ ]:
filas = []
for decay in (0.0, 0.2, 1.0):
    Xd, Qd = genera_vectores(N, D, decay=decay, seed=1)
    gt_d = exacto_topk(Qd, Xd, 10)
    ivf_ = faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, NLIST, faiss.METRIC_INNER_PRODUCT); ivf_.train(Xd); ivf_.add(Xd); ivf_.nprobe = 8
    hn_ = faiss.IndexHNSWFlat(D, 16, faiss.METRIC_INNER_PRODUCT); hn_.hnsw.efConstruction = 100; hn_.add(Xd); hn_.hnsw.efSearch = 32
    pq_ = faiss.IndexIVFPQ(faiss.IndexFlatIP(D), D, NLIST, 16, 8, faiss.METRIC_INNER_PRODUCT); pq_.train(Xd); pq_.add(Xd); pq_.nprobe = 8
    filas.append(dict(decay=decay, IVF_nprobe8=recall_vs_gt(ivf_.search(Qd, 10)[1], gt_d), HNSW_ef32=recall_vs_gt(hn_.search(Qd, 10)[1], gt_d),
                      IVFPQ16_nprobe8=recall_vs_gt(pq_.search(Qd, 10)[1], gt_d)))
sens = pd.DataFrame(filas).set_index("decay")
display(sens.round(3))
sens.plot(kind="bar", figsize=(7, 3.4), color=["#1F6FEB", "#F28E2B", "#2CA02C"]); plt.ylabel("recall@10"); plt.title("Mismos parámetros, datos distintos"); plt.xticks(rotation=0); plt.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con la **misma** configuración el recall cambia mucho entre conjuntos: IVF y HNSW **mejoran** cuando los datos son más anisótropos (los vecinos se separan mejor), mientras IVF-PQ **empeora** (las diferencias entre vecinos son pequeñas frente al error de cuantización; sin *refine* se nota). Los parámetros no se transfieren entre datasets. Un barrido hecho con vectores sintéticos o de otro modelo **no sirve** para fijar los parámetros de producción: vuelve a medir con tus embeddings (esto es lo que hace el proyecto con los vectores de Políglota).

## 11 · Puente con `recsys-course/08_deep_retrieval_ann`
| Tema | Aquí (módulo 14) | Allí (recsys 08) |
|---|---|---|
| Vectores | embeddings de texto (Matryoshka, binarios…) | embeddings de ítems de un *two-tower* |
| Índices | desde cero: IVF, PQ/OPQ, HNSW, Vamana, RaBitQ | uso de FAISS: Flat/IVF/IVF-PQ/HNSW, hnswlib, ScaNN |
| Foco | algoritmos, cuantización con garantías, disco, filtros | integración con el modelo, servicio top-K, filtrado de negocio |
Léelo para ver cómo el mismo índice sirve a un *two-tower*; lo que cambia es el **modelo que produce los vectores**, no el índice.

## 12 · 🏭 En producción

- **FAISS** (Meta; Douze et al., 2024) es la biblioteca de referencia: Flat, IVF, PQ/OPQ, HNSW, cuantización escalar, GPU y composición de índices con `index_factory`. El paper documenta el diseño y las decisiones de ingeniería.
- **HNSW** es el índice por defecto de gran parte de los motores: **Lucene/Elasticsearch/OpenSearch**, **pgvector** (ofrece HNSW e IVFFlat), Qdrant y Weaviate. Por eso la mayoría de las decisiones del equipo son `M`, `efConstruction` y `efSearch`.
- **Cuantización binaria derivada de RaBitQ:** Elastic documenta BBQ (Elasticsearch 8.16, Lucene) con reducciones de memoria de ≈ 95 % según su propio benchmark; Milvus informa soporte de RaBitQ desde la versión 2.6.
- **DiskANN** (Microsoft, código abierto) para índices que no caben en RAM; **ScaNN** (Google, código abierto) para MIPS con cuantización anisótropa.
- **Multivector:** Qdrant recomienda desactivar HNSW en el campo multivector cuando solo se usa para *reranking* (módulo 13).
- **Evaluación independiente:** *ann-benchmarks* (Aumüller et al., 2018) compara implementaciones en curvas recall-QPS; úsalo como referencia para tu clase de dataset, no como sustituto de medir los tuyos.
- La **operación** (sharding, réplicas, actualizaciones, backups) es el tema del módulo 18.

## 13 · 🧠 Secretos de la élite

1. **Mide con el protocolo correcto.** Neighbor recall contra fuerza bruta **sobre queries reales**, batch=1, un hilo, p99 (no la media) y bytes serializados reales. Comparar un índice a 8 hilos en lote con otro a 1 hilo es la fuente de la mitad de los benchmarks engañosos.
2. **Los parámetros no se transfieren entre datasets** (lo viste en la sección 10): el `nprobe` o `efSearch` que da 0.95 con un modelo de embeddings da otro valor con el siguiente. Reajusta tras **cada cambio de embedder, de dimensión o de cuantización** (módulo 12).
3. **Primero reduce el coste de la distancia, luego el número de distancias.** Matryoshka (módulo 12) divide el coste de cada distancia por 2–4× *antes* de que el índice haga nada, y se combina con cualquier ANN.
4. **PQ casi nunca va sola.** El patrón de producción es «códigos comprimidos en RAM para generar candidatos + vectores completos (disco/SSD) para re-ordenar» (IVF-PQ + refine, DiskANN, binaria + rescoring).
5. **HNSW cuesta construirlo y mantenerlo.** Construir es lento (aquí, decenas de veces más que IVF), consume más RAM que los vectores solos y los borrados suelen implementarse como marcas (*tombstones*) que degradan el grafo hasta reconstruirlo; planifica reconstrucciones periódicas (módulo 18). FreshDiskANN existe precisamente para el caso de actualización continua.
6. **Vigila la cola, no la media.** En IVF, listas muy desiguales (aquí la mayor tiene ≈ 3× la media) disparan el p99; en HNSW, la cola depende de `efSearch` y de la frialdad de la caché. Re-balancea o usa más celdas.
7. **Filtros: elige la estrategia según la selectividad.** Pre-filtro exacto cuando `s·N` es pequeño, filtro en el grafo cuando es moderado y post-filtro solo si es alto. Si cada inquilino (*tenant*) tiene pocos vectores, un índice por tenant puede ser mejor que un filtro.
8. **La binaria mejora con la dimensión.** Con RaBitQ el error baja como $1/\sqrt D$: para embeddings de 768–3072 dimensiones la binaria es mucho más viable que a 64 (por eso funciona bien con modelos recientes y mal con embeddings pequeños).
9. **El *refine* es un dial barato:** subir `k_factor` (candidatos re-puntuados) cuesta lecturas del vector completo, no distancias aproximadas; es la primera palanca cuando el recall no llega al SLA.
10. **Recall del índice ≠ recall de la tarea.** Un índice con recall@10 = 0.95 sobre un embedder con nDCG@10 = 0.60 sigue dando ≈ 0.57 de nDCG: evalúa ambas capas por separado (módulo 15).

## 14 · ⚠️ Errores comunes

| Error | Síntoma | Arreglo |
|---|---|---|
| Medir latencia en lote o con varios hilos | cifras irreproducibles en producción | batch=1, 1 hilo, p50/p99 |
| Calcular recall contra el propio índice aproximado | recall siempre ≈ 1 | ground truth con fuerza bruta exacta |
| Evaluar con queries tomadas del corpus de índice | recall inflado (el vecino más cercano es la propia query) | queries separadas del índice |
| Copiar parámetros de otro dataset/modelo | recall inesperadamente bajo | re-barrer con tus vectores |
| Usar L2 con vectores sin normalizar y producto interno en el índice | ranking incoherente | normalizar o elegir la métrica adecuada |
| PQ con `m` que no divide a `d` o entrenada con pocas muestras | libros pobres | `d % m == 0`, ≥ 39·256 muestras por subespacio como regla de FAISS |
| HNSW con `efSearch < k` | menos resultados que `k` | `efSearch ≥ k` |
| Post-filtro con filtros muy selectivos | resultados vacíos | pre-filtro / más sobremuestreo / grafo filtrable |
| Olvidar el *refine* en IVF-PQ | recall estancado | `IndexRefineFlat` o re-ranking con vectores completos |
| Mezclar vectores de dos versiones de embedder en un índice | recall aleatorio | un índice por (modelo, versión, dimensión, precisión) |

## 15 · Autoevaluación

<details><summary><b>1 · (Cálculo)</b> HNSW con $N=10^8$, $d=768$ en <code>float32</code> y $M=32$ (grado máximo $2M=64$ en la capa 0, enlaces de 4 B). Estima la RAM del índice.</summary>

Vectores: $10^8\cdot768\cdot4=307.2$ GB. Capa 0: $10^8\cdot64\cdot4=25.6$ GB (cota superior; las capas superiores añaden ≈ $1/M$ de nodos). Total ≈ **333 GB** (más metadatos). El grafo añade ≈ 8–10 % al tamaño de los vectores: el cuello de botella son los **vectores**, por eso se comprimen (PQ, binaria) o se mueven a disco.
</details>

<details><summary><b>2 · (Cálculo)</b> IVF con $N=10^7$, $n_{list}=4096$, $n_{probe}=32$. ¿Qué fracción del corpus se escanea y cuántas comparaciones hace una query?</summary>

Fracción $=32/4096=0.78\%$ → ≈ 78 000 vectores (si las listas están equilibradas) + 4096 comparaciones con centroides ≈ **82 000** distancias frente a $10^7$ de la fuerza bruta (≈ 120× menos).
</details>

<details><summary><b>3 · (Cálculo)</b> PQ con $d=768$, $m=96$, 8 bits por código: ¿bytes por vector, factor de compresión y tamaño de las tablas ADC por query?</summary>

96 B por vector; $768\cdot4/96=32\times$. Tablas: $96\times256$ floats $=98\,304$ B ≈ 96 KB por query (se calculan una vez por query y por celda si se usa el residuo).
</details>

<details><summary><b>4 · (Diagnóstico)</b> Tu HNSW dio recall@10 = 0.98 en validación, pero a las 3 semanas de producción cae a 0.90 sin cambiar nada. ¿Qué miras?</summary>

(a) Entraron muchos documentos nuevos con borrados por marca: el grafo se degrada (reconstruir). (b) **Deriva de la distribución de queries o de documentos** respecto a la de validación. (c) Cambió la versión del embedder para parte del índice. (d) Un cambio de `efSearch` por tuning de latencia. Revisa con una muestra de queries actuales y su ground truth por fuerza bruta.
</details>

<details><summary><b>5 · (Diagnóstico)</b> En IVF-Flat el p50 es 3 ms pero el p99 es 40 ms. ¿Causas probables?</summary>

Listas de tamaños muy desiguales (algunas queries caen en celdas enormes), caché fría, `nprobe` alto con contención de hilos, y GC/páginas de memoria. Soluciones: más celdas/balanceo, reducir `nprobe` con *refine*, fijar afinidad de hilos y precalentar.
</details>

<details><summary><b>6 · (Transferencia)</b> Tienes $10^9$ vectores de 768 dims y un servidor con 64 GB de RAM y SSD NVMe. Diseña el índice.</summary>

HNSW no cabe (≈ 3 TB). Opciones: **DiskANN/Vamana** (PQ de, p. ej., 64–96 B/vector en RAM = 64–96 GB, aún demasiado → PQ de 32 B = 32 GB + grafo y vectores completos en SSD), o **IVF-PQ + refine** con vectores completos en disco, o **binaria (RaBitQ) 96 B/vector + rescoring**. Reduciría la dimensión con Matryoshka (módulo 12) para bajar el PQ. Validaría recall y p99 con 1 000 queries reales y ground truth en una submuestra.
</details>

<details><summary><b>7 · (Transferencia)</b> ¿Por qué RaBitQ divide por $\langle\bar o,o\rangle$ y qué ocurre si no lo hace?</summary>

Porque el vector cuantizado $\bar o$ no es unitario respecto a $o$: $\langle\bar o,q\rangle$ **subestima** sistemáticamente $\langle o,q\rangle$ por el factor $\langle\bar o,o\rangle<1$. Dividir da un estimador **insesgado**; sin ello, los productos internos altos (los de los vecinos que buscamos) salen comprimidos y el ranking entre candidatos se distorsiona (en la sección 8 el ingenuo estimaba ≈ 0.75 del valor real).
</details>

## 16 · Referencias (verificadas; consultadas en oct-2026)

**Papers (arXiv)**
- Malkov & Yashunin (2016/2018). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs*. <https://arxiv.org/abs/1603.09320>
- Guo et al. (2019). *Accelerating Large-Scale Inference with Anisotropic Vector Quantization* (ScaNN). <https://arxiv.org/abs/1908.10396>
- Gao & Long (2024). *RaBitQ: Quantizing High-Dimensional Vectors with a Theoretical Error Bound for Approximate Nearest Neighbor Search*. <https://arxiv.org/abs/2405.12497> · Gao et al. (2024). *Practical and Asymptotically Optimal Quantization of High-Dimensional Vectors in Euclidean Space for Approximate Nearest Neighbor Search* (Extended RaBitQ). <https://arxiv.org/abs/2409.09913>
- Douze et al. (2024). *The Faiss library*. <https://arxiv.org/abs/2401.08281>
- Aumüller, Bernhardsson & Faithfull (2018). *ANN-Benchmarks: A Benchmarking Tool for Approximate Nearest Neighbor Algorithms*. <https://arxiv.org/abs/1807.05614>
- Patel et al. (2024). *ACORN: Performant and Predicate-Agnostic Search Over Vector Embeddings and Structured Data*. <https://arxiv.org/abs/2403.04871>
- Singh et al. (2021). *FreshDiskANN: A Fast and Accurate Graph-Based ANN Index for Streaming Similarity Search*. <https://arxiv.org/abs/2105.09613>

**Papers (sin arXiv)**
- Jégou, Douze & Schmid (2011). *Product Quantization for Nearest Neighbor Search*. IEEE TPAMI.
- Ge, He, Ke & Sun (2013). *Optimized Product Quantization for Approximate Nearest Neighbor Search*. CVPR.
- Subramanya et al. (2019). *DiskANN: Fast Accurate Billion-point Nearest Neighbor Search on a Single Node*. NeurIPS.

**Código y blogs**
- FAISS: <https://github.com/facebookresearch/faiss> · DiskANN: <https://github.com/microsoft/DiskANN> · ScaNN: <https://github.com/google-research/google-research/tree/master/scann>
- Elastic Search Labs. *Better Binary Quantization (BBQ) in Lucene and Elasticsearch*. <https://www.elastic.co/search-labs/blog/better-binary-quantization-lucene-elasticsearch>
- Milvus. *Bring Vector Compression to the Extreme: How Milvus Serves 3× More Queries with RaBitQ* (blog de Milvus).
- Curso de recsys del repositorio: [`recsys-course/08_deep_retrieval_ann`](../../recsys-course/08_deep_retrieval_ann/README.md).

**Para profundizar:** módulo 15 (evaluación de retrieval de extremo a extremo), módulo 18 (operación de bases vectoriales: Qdrant/pgvector, versionado y migración).